# Sensitivity Analysis (LLMs) — Main Notebook

Hypothesis: The underlying LLM of an MAAI statistically significantly affects its fairness judgments; specifically, Group 1 and Group 2 LLMs differ in their judgments.

This notebook prepares two groups of aligned experiment configurations (Group 1, Group 2),
runs them selectively in parallel, and compares the outcome distributions across groups (5 categories).

- 34 configs per group (default), total 68.
- Per-config: a shared temperature drawn from U(0, 1.5), shared random seed, and agent models selected per group.
- Language is English for all runs.
- Utility agent is `gemini-2.5-flash`.
- Voting detection mode is set to "complex" for all runs.
- Exact test: Fisher–Freeman–Halton via R (if available). No Chi-square fallback.
- Effect size: Cramér's V (bias-corrected, as in Hypothesis 1).


## Prerequisites

1. **Environment**: Ensure the virtual environment is activated with all dependencies installed.
2. **API Keys**: Set up `.env` with required API keys (see `CLAUDE.md` for details).
3. **Configuration**: Review the execution control flags in the first code cell.

## How to Use

1. Set `create_new_configurations = True` to generate new experiment configs
2. Set `run_experiment = True` to execute experiments
3. Run all cells in order

> **Note**: Both flags default to `False` to prevent accidental execution.

In [ ]:
# Flag to run the experiment and create new configurations
run_experiment = False
create_new_configurations = False


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import sys, os
from pathlib import Path

# Ensure repo root on sys.path (for local package imports)
def _add_repo_root_to_sys_path():
    here = Path.cwd().resolve()
    for p in [here] + list(here.parents):
        if (p / 'main.py').exists() and (p / 'experiment_execution').is_dir():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
    return here
_REPO_ROOT = _add_repo_root_to_sys_path()

import json
import random
import shutil
import yaml
from collections import Counter
from experiment_execution.utils_experiment_execution.runner import (
    list_config_files,
    select_configs,
    run_configs_in_parallel,
)


## 2. Model Selection Process

In [ ]:
# 2. Model Definition

# We define two groups of models for the sensitivity analysis (Group 1 and Group 2).

# Group 1 Models 
GROUP_1_MODELS = [
    "openai/gpt-oss-120b",
    "x-ai/grok-4-fast",
    "x-ai/grok-code-fast-1"
]

# Group 2 Models
GROUP_2_MODELS = [
    "deepseek/deepseek-v3.2-exp",
    "qwen/qwen3-next-80b-a3b-thinking",
    "z-ai/glm-4.5-air"
]


In [ ]:
# Base paths
BASE_DIR = _REPO_ROOT / 'experiment_execution' / 'sensitivity_analysis_llm'
CONFIGS_BASE = BASE_DIR / 'configs'
LOGS_BASE = BASE_DIR / 'terminal_outputs'
RESULTS_BASE = BASE_DIR / 'results'
TRANSCRIPTS_BASE = BASE_DIR / 'transcripts'

GROUP_SETS = {
    'group_1': GROUP_1_MODELS,
    'group_2': GROUP_2_MODELS,
}

# Ensure subfolders exist
for key in GROUP_SETS.keys():
    (CONFIGS_BASE / key).mkdir(parents=True, exist_ok=True)
    (LOGS_BASE / key).mkdir(parents=True, exist_ok=True)
    (RESULTS_BASE / key).mkdir(parents=True, exist_ok=True)
    (TRANSCRIPTS_BASE / key).mkdir(parents=True, exist_ok=True)

CONFIGS_BASE, LOGS_BASE, RESULTS_BASE, TRANSCRIPTS_BASE


## 3. Generate Configurations


In [ ]:
INCOME_CLASS_PROBS = {
    'high': 0.05,
    'medium_high': 0.10,
    'medium': 0.50,
    'medium_low': 0.25,
    'low': 0.10,
}

def make_agents_with_models(temp: float, models: list[str]) -> list[dict]:
    agents = []
    for i in range(0, 5):
        agents.append({
            'name': f'Agent_{i}',
            'personality': 'You are an American college student',
            'model': models[i],
            'temperature': float(temp),
            'memory_character_limit': 25000,
            'reasoning_enabled': True,
        })
    return agents

def build_config(group_key: str, temp: float, seed_val: int, models: list[str]) -> dict:
    return {
        'language': 'English',
        'seed': int(seed_val),
        'agents': make_agents_with_models(temp, models),
        'utility_agent_model': "gemini-2.5-flash",
        'utility_agent_temperature': 0.0,
        'phase2_rounds': 10,
        'distribution_range_phase2': [2, 6],
        'income_class_probabilities': INCOME_CLASS_PROBS,
        'original_values_mode': { 'enabled': True },
        'voting_mode': 'complex',
    }

GLOBAL_SEED = 12345
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

def generate_aligned_configs(n: int = 34) -> dict[str, list[Path]]:
    paths: dict[str, list[Path]] = {k: [] for k in GROUP_SETS.keys()}
    for idx in range(1, n + 1):
        temp = random.uniform(0.0, 1.5)  # shared per condition across groups
        seed_val = random.randint(0, 2**31 - 1)
        
        # Select models for this config
        # For each group, randomly sample 5 models from the group list (with replacement)
        models_per_group = {}
        for g_key, g_models in GROUP_SETS.items():
             models_per_group[g_key] = [random.choice(g_models) for _ in range(5)]

        for g_key, g_models in models_per_group.items():
            cfg = build_config(group_key=g_key, temp=temp, seed_val=seed_val, models=g_models)
            out_dir = (CONFIGS_BASE / g_key)
            out_dir.mkdir(parents=True, exist_ok=True)
            fname = out_dir / f'sensitivity_analysis_llm_{g_key}_condition_{idx}_config.yaml'
            with open(fname, 'w') as f:
                yaml.safe_dump(cfg, f, sort_keys=False)
            paths[g_key].append(fname)
    return paths


In [ ]:
if create_new_configurations:
    files_by_group = generate_aligned_configs(n=34)
    print({k: len(v) for k, v in files_by_group.items()})


## 4. Run Configurations


In [ ]:
def run_group_set(group_key: str, include_indices=None, include_names=None, concurrency: int = 4, timeout_sec: int | None = None):
    cfg_dir = CONFIGS_BASE / group_key
    logs_dir = LOGS_BASE / group_key
    results_dir = RESULTS_BASE / group_key
    configs = list_config_files(cfg_dir)

    selected = select_configs(configs, include_indices=include_indices, include_names=include_names)
    print(f'[{group_key}] Found {len(configs)} configs; selected {len(selected)}')

    run_results = run_configs_in_parallel(
        selected,
        concurrency=concurrency,
        logs_dir=logs_dir,
        results_dir=results_dir,
        timeout_sec=timeout_sec,
    )
    ok = sum(1 for r in run_results if r.get('ok'))
    print(f'[{group_key}] Completed: {ok}/{len(run_results)} OK')
    return run_results


In [ ]:
if run_experiment:
    all_run_results = []
    import time

    for group_key in ['group_1', 'group_2']:
        print(f'\n=== Running ALL {group_key.upper()} conditions (34 configs) ===')
        start_time = time.time()
        run_results = run_group_set(
            group_key,
            include_indices=list(range(1, 35)),
            concurrency=2,
            timeout_sec=None
        )
        duration = time.time() - start_time
        print(f'Completed all {group_key.upper()} conditions in {duration:.1f}s')
        all_run_results.extend(run_results)


## 5. Analysis


In [ ]:
CATEGORIES = [
    'maximizing_floor',
    'maximizing_average',
    'maximizing_average_floor_constraint',
    'maximizing_average_range_constraint',
    'disagreement',
]

def categorize_result(result_path: Path) -> str:
    try:
        with open(result_path, 'r') as f:
            data = json.load(f)
        gi = data.get('general_information', {})
        consensus = gi.get('consensus_reached', False)
        principle = gi.get('consensus_principle')
        if consensus and principle in CATEGORIES:
            return principle
        return 'disagreement'
    except Exception:
        return 'disagreement'

def count_by_group() -> dict[str, Counter]:
    out: dict[str, Counter] = {}
    for k in GROUP_SETS.keys():
        counts = Counter()
        result_files = sorted((RESULTS_BASE / k).glob('*_results.json'))
        for rp in result_files:
            counts[categorize_result(rp)] += 1
        for cat in CATEGORIES:
            counts.setdefault(cat, 0)
        out[k] = counts
    return out

group_counts = count_by_group()
for k, counts in group_counts.items():
    print(f'{k} counts:', dict(counts))

# Build contingency table: rows=categories, cols=[group_1, group_2]
col_order = ['group_1', 'group_2']
contingency = np.vstack([[group_counts[col][cat] for col in col_order] for cat in CATEGORIES])
contingency, CATEGORIES, col_order


In [ ]:
def fisher_freeman_halton_pvalue_r(contingency: np.ndarray) -> float | None:
    if shutil.which('Rscript') is None:
        return None
    r_matrix = ','.join(str(int(x)) for x in contingency.flatten(order='C'))
    nrow, ncol = contingency.shape
    r_code = f"""m <- matrix(c({r_matrix}), nrow={nrow}, ncol={ncol}, byrow=TRUE);
f <- tryCatch(fisher.test(m), error=function(e) NA);
if (is.list(f)) {{ cat(f$p.value) }} else {{ cat('NA') }}
"""
    import subprocess
    try:
        out = subprocess.check_output(['Rscript', '-e', r_code], stderr=subprocess.STDOUT, text=True, timeout=30)
        out = out.strip()
        return float(out) if out and out != 'NA' else None
    except subprocess.TimeoutExpired:
        return None
    except Exception:
        return None

p_ffh = fisher_freeman_halton_pvalue_r(contingency)
print(f'Fisher-Freeman-Halton exact test p-value: {p_ffh}')

from experiment_execution.utils_experiment_execution import bias_corrected_cramers_v

v_corr = bias_corrected_cramers_v(contingency)
print(f"Cramér's V (bias-corrected): {v_corr:.4f}")
